# PiShield — a Shield Layer for QFLRA requirements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/hierarchical-requirements/examples/shield_layer_qflra.ipynb)

This notebook runs **end-to-end with no external downloads**. It is the sibling of the linear example, but uses **QFLRA** requirements — *quantifier-free linear real arithmetic*: linear inequalities combined with **disjunctions** (`or`) and negations. Disjunctions let us state a **non-convex** feasible region — a union of pieces — which the purely conjunctive linear backend cannot express.

The showcase is **three parallel bands** on the gap $g = y_0 - y_1$:

$$g \;\in\; [0,\,0.1] \;\cup\; [0.3,\,0.4] \;\cup\; [0.6,\,0.7]$$

The Shield Layer sends any prediction to the **nearest** of the three bands, leaves predictions that are already feasible untouched, and can be used both **at inference** (correct a trained model) and **during training** (backprop straight through it).

## Setup

In [ ]:
import importlib.util, subprocess, sys, os

root = os.path.abspath(os.getcwd())
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'pishield')):
    root = os.path.dirname(root)
if os.path.isdir(os.path.join(root, 'pishield')):
    sys.path.insert(0, root)
    print('Using local PiShield at', root)
elif importlib.util.find_spec('pishield') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/mihaela-stoian/PiShield.git@hierarchical-requirements'],
                   check=True)
    print('Installed PiShield')
else:
    print('PiShield already available')

In [ ]:
import contextlib, io
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from pishield.shield_layer import build_shield_layer

torch.manual_seed(0)

def quiet_build(*args, **kwargs):
    """build_shield_layer, but without the backend's verbose constraint dump."""
    with contextlib.redirect_stdout(io.StringIO()):
        return build_shield_layer(*args, **kwargs)

# The three bands are defined on the gap  g = y_0 - y_1.
BANDS = [(0.0, 0.1), (0.3, 0.4), (0.6, 0.7)]
TOL = 1e-2   # the QFLRA backend certifies satisfaction at this tolerance

GREEN, BLUE, RED, GREY = "#5a9e4a", "#3f72c4", "#d0433a", "#8a8a8a"
lo, hi = -0.05, 1.05

def draw_bands(ax):
    """Shade the three feasible diagonal bands in the (y_0, y_1) plane."""
    for a, b in BANDS:
        ax.fill_between([lo, hi], [lo - a, hi - a], [lo - b, hi - b], color=GREEN, alpha=0.12, lw=0, zorder=0)
        ax.plot([lo, hi], [lo - a, hi - a], "--", color="#999", lw=.7, zorder=1)
        ax.plot([lo, hi], [lo - b, hi - b], "--", color="#999", lw=.7, zorder=1)

def band_of(gap):
    """Band index (0,1,2) each gap falls in, or -1 if in none."""
    idx = torch.full(gap.shape, -1, dtype=torch.long)
    for i, (a, b) in enumerate(BANDS):
        idx[(gap >= a - TOL) & (gap <= b + TOL)] = i
    return idx

def per_band(out):
    g = out[:, 0] - out[:, 1]
    return [int((band_of(g) == i).sum()) for i in range(3)]

## 1. Write the requirements file

A union of three bands is written as the overall range `0 <= g <= 0.7` **minus two forbidden middle strips**. Each forbidden strip is a disjunction — *below it OR above it* — and it is the `or` that makes these QFLRA rather than linear requirements.

In [ ]:
# Feasible region for the gap g = y_0 - y_1:   [0,0.1] U [0.3,0.4] U [0.6,0.7]
# Written as: 0 <= g <= 0.7, minus the two forbidden middle strips.
# Each forbidden strip is a DISJUNCTION ("or") -- this is what makes it QFLRA,
# not plain linear.
requirements_path = "qflra_bands.txt"
with open(requirements_path, "w") as f:
    f.write(
        "ordering y_0 y_1\n"
        "y_0 - y_1 >= 0\n"                                 # g >= 0
        "y_1 - y_0 >= -0.7\n"                              # g <= 0.7
        "y_1 - y_0 >= -0.1 or y_0 - y_1 >= 0.3\n"          # g <= 0.1  OR  g >= 0.3
        "y_1 - y_0 >= -0.4 or y_0 - y_1 >= 0.6\n")         # g <= 0.4  OR  g >= 0.6

print(open(requirements_path).read())

# build_shield_layer auto-detects the type: it reads the "or" disjunctions and
# picks the QFLRA backend on its own -- no need to name it explicitly.
shield = quiet_build(2, requirements_path)

## 2. What the layer does

We feed the layer a cloud of arbitrary points. Each one is moved to the **nearest** of the three bands, so all three fill up and no point is left in a forbidden zone. And — the key property of a refinement layer — points that are **already feasible are returned unchanged**.

In [ ]:
cloud = torch.rand(1500, 2)
projected = shield(cloud.clone())
g = projected[:, 0] - projected[:, 1]
print("points landing in each band:", per_band(projected))
print("points in NO band (violations):", int((band_of(g) == -1).sum()), "/", len(cloud))

# Idempotency: points that are already feasible must be returned unchanged.
feasible = projected.clone()                     # everything above is now feasible
assert torch.allclose(shield(feasible.clone()), feasible, atol=1e-4)
print("re-applying the shield to feasible points changes them by at most",
      float((shield(feasible.clone()) - feasible).abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.8), sharex=True, sharey=True)
draw_bands(axes[0])
axes[0].scatter(cloud[:, 0], cloud[:, 1], s=9, c=GREY, alpha=.4, edgecolors="none", zorder=2)
axes[0].set_title("Arbitrary predictions")
draw_bands(axes[1])
axes[1].scatter(projected[:, 0], projected[:, 1], s=9, c=GREEN, alpha=.5, edgecolors="none", zorder=2)
axes[1].set_title(f"After PiShield  (per band: {per_band(projected)})")
for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("$y_0$"); ax.grid(True, ls=":", alpha=.3)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("$y_1$")
fig.suptitle("QFLRA: gap $y_0-y_1 \\in [0,0.1]\\cup[0.3,0.4]\\cup[0.6,0.7]$  (a non-convex region)", y=1.01)
plt.tight_layout(); plt.show()

## 3. A closer look: where does each gap go?

Sweeping the input gap from below the first band to above the third, the returned gap steps cleanly through **all three** bands: values inside a band are kept as-is (the staircase sits on the diagonal there), and values in a forbidden zone jump to whichever band edge is closer.

In [ ]:
gin = torch.linspace(-0.05, 0.85, 181)
inp = torch.stack([gin, torch.zeros_like(gin)], dim=1)
gout = shield(inp.clone())[:, 0] - shield(inp.clone())[:, 1]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for a, b in BANDS:
    ax.axhspan(a, b, color=GREEN, alpha=0.14, lw=0)
ax.plot(gin, gin, ls=":", color=GREY, lw=1, label="identity (no correction)")
ax.plot(gin, gout, color=BLUE, lw=2, label="gap returned by PiShield")
ax.set_xlabel("input gap  $y_0 - y_1$"); ax.set_ylabel("output gap")
ax.grid(True, ls=":", alpha=.4); ax.legend(loc="upper left", framealpha=.9)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.set_title("Each gap is sent to the nearest band (all three are reachable)")
plt.tight_layout(); plt.show()

## 4. A learning task

A small regression whose targets are spread over all three bands — both the level and *which band* are functions of the inputs, so the task is learnable.

In [ ]:
N, in_dim = 900, 4
X = torch.randn(N, in_dim)
w_level, w_band = torch.randn(in_dim, 1), torch.randn(in_dim, 1)

center = torch.sigmoid(X @ w_level).squeeze() * 0.3 + 0.35    # a learnable level in [0.35, 0.65]
# which band each target belongs to is also a (learnable) function of the inputs
score = (X @ w_band).squeeze()
q1, q2 = torch.quantile(score, torch.tensor([1 / 3, 2 / 3]))
choice = (score > q1).long() + (score > q2).long()           # 0, 1 or 2, ~balanced
band_mid = torch.tensor([(a + b) / 2 for a, b in BANDS])
gap = band_mid[choice] + (torch.rand(N) - 0.5) * 0.06
Y = torch.stack([center + gap / 2, center - gap / 2], dim=1)

print("targets per band:", [int((choice == i).sum()) for i in range(3)])

fig, ax = plt.subplots(figsize=(4.9, 4.7))
draw_bands(ax)
ax.scatter(Y[:, 0], Y[:, 1], s=12, c=GREEN, alpha=.5, edgecolors="white", linewidths=.25, zorder=2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("$y_0$"); ax.set_ylabel("$y_1$"); ax.grid(True, ls=":", alpha=.3)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.set_title("Training data spread over all three bands")
plt.tight_layout(); plt.show()

## 5. Training with the Shield Layer

We train two networks, **each with its own progress bar**: a plain MLP with no constraints, and the same MLP with the Shield Layer inside `forward`, so it is trained *through* the layer.

The unconstrained network leaves many of its predictions in the forbidden zones between the bands (shown in **red** below); the network trained through PiShield is guaranteed to land in a band on every forward pass. Training through a hard, non-convex layer, it does tend to *lean on* the layer and over-use the middle band — a higher MSE and a more lopsided per-band count — but it never violates the constraint.

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Linear(in_dim, 16), nn.Tanh(), nn.Linear(16, 2))

class ShieldedMLP(nn.Module):
    """The MLP with the QFLRA Shield Layer inside forward -- so it is trained *through* it."""
    def __init__(self):
        super().__init__()
        self.model = make_mlp()
        self.shield = quiet_build(2, requirements_path)
    def forward(self, x):
        return self.shield(self.model(x))

def train(model, label, epochs=300, lr=5e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in tqdm(range(epochs), desc=f"training {label}"):
        opt.zero_grad(); loss = loss_fn(model(X), Y); loss.backward(); opt.step()
    return model

# an unconstrained network, and the same network trained *through* the Shield Layer
plain = train(make_mlp(), "NN (no PiShield)")
through = train(ShieldedMLP(), "PiShield (included at training)")

with torch.no_grad():
    plain_out = plain(X)      # unconstrained
    thru_out = through(X)     # PiShield used during training

def mse(o):
    return ((o - Y) ** 2).mean().item()

def num_violations(o):
    return int((band_of(o[:, 0] - o[:, 1]) == -1).sum())

print(f"\ntargets: per band = {[int((choice == i).sum()) for i in range(3)]}\n")
for name, o in [("NN (no PiShield)", plain_out),
                ("PiShield (included at training)", thru_out)]:
    print(f"{name:<33}: MSE = {mse(o):.4f}, violations = {num_violations(o):>3}/{N}, per band = {per_band(o)}")

## 6. Visualising the result

Real data fills all three bands. The raw network scatters across the forbidden zones. Applying PiShield at inference pulls those strays into the nearest band while keeping the spread; the trained-through network is also feasible everywhere but concentrates more on the middle band.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.7), sharex=True, sharey=True)

# real data
draw_bands(axes[0])
axes[0].scatter(Y[:, 0], Y[:, 1], s=12, c=GREEN, alpha=.5, edgecolors="none", zorder=2)
axes[0].set_title("Real data")

# unconstrained NN -- points that violate the constraint are shown in red
bad = (band_of(plain_out[:, 0] - plain_out[:, 1]) == -1).numpy()
pp = plain_out.numpy()
draw_bands(axes[1])
axes[1].scatter(pp[~bad, 0], pp[~bad, 1], s=12, c=BLUE, alpha=.45, edgecolors="none", zorder=2, label="in a band")
axes[1].scatter(pp[bad, 0], pp[bad, 1], s=16, c=RED, alpha=.7, edgecolors="none", zorder=3,
                label=f"violates ({int(bad.sum())})")
axes[1].set_title("NN (no PiShield)"); axes[1].legend(loc="upper left", fontsize=8, framealpha=.9)

# trained through the shield -- feasible by construction
draw_bands(axes[2])
axes[2].scatter(thru_out[:, 0], thru_out[:, 1], s=12, c=GREEN, alpha=.5, edgecolors="none", zorder=2)
axes[2].set_title("PiShield (included at training)")

for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.set_xlabel("$y_0$")
    ax.grid(True, ls=":", alpha=.3)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("$y_1$")
plt.tight_layout(); plt.show()

## 7. Takeaway

QFLRA requirements express genuinely **non-convex** constraints — a union of regions — that the linear backend cannot, and the Shield Layer enforces them exactly: every output is inside a band, already-feasible points are untouched, and violations are snapped to the nearest band. It works both as an inference-time projection and as a differentiable layer you train through, and `build_shield_layer` auto-detects the QFLRA backend from the disjunctions. One sharp edge to keep in mind: satisfaction is certified at a `1e-2` tolerance rather than machine precision.